# V0 — End-to-End Retrieval Pipeline

Recipe embeddings via sentence-transformers, user embeddings as weighted means of liked recipes, cosine similarity retrieval, evaluated on a temporal holdout.

Dataset: Food.com (shuyangli94/food-com-recipes-and-user-interactions)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

## 1. Load and inspect data

In [2]:
recipes = pd.read_csv(DATA_DIR / "RAW_recipes.csv")
interactions = pd.read_csv(DATA_DIR / "RAW_interactions.csv")

print(f"Recipes: {len(recipes):,}")
print(f"Interactions: {len(interactions):,}")
print(f"Unique users: {interactions['user_id'].nunique():,}")
print(f"\nRecipe columns: {recipes.columns.tolist()}")
print(f"Interaction columns: {interactions.columns.tolist()}")
recipes.head(2)

Recipes: 231,637
Interactions: 1,132,367
Unique users: 226,570

Recipe columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients']
Interaction columns: ['user_id', 'recipe_id', 'date', 'rating', 'review']


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6


In [3]:
# Check what tags look like — these are our concept vocabulary for V0
print(recipes['tags'].iloc[0])
print(type(recipes['tags'].iloc[0]))

['60-minutes-or-less', 'time-to-make', 'course', 'main-ingredient', 'cuisine', 'preparation', 'occasion', 'north-american', 'side-dishes', 'vegetables', 'mexican', 'easy', 'fall', 'holiday-event', 'vegetarian', 'winter', 'dietary', 'christmas', 'seasonal', 'squash']
<class 'str'>


## 2. Prepare recipe text for embedding

Each recipe gets a single text string: name + ingredients + tags.
sentence-transformers will encode this into a dense vector.

In [4]:
import ast

def parse_list_str(s):
    """Parse a string representation of a list into an actual list."""
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return []

recipes['tags_list'] = recipes['tags'].apply(parse_list_str)
recipes['ingredients_list'] = recipes['ingredients'].apply(parse_list_str)

def build_recipe_text(row):
    """Combine name, ingredients, and tags into a single embedding input."""
    name = row['name']
    ingredients = ', '.join(row['ingredients_list'])
    tags = ', '.join(row['tags_list'])
    return f"{name}. Ingredients: {ingredients}. Tags: {tags}"

recipes['text'] = recipes.apply(build_recipe_text, axis=1)
print(recipes['text'].iloc[0])

arriba   baked winter squash mexican style. Ingredients: winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt. Tags: 60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, side-dishes, vegetables, mexican, easy, fall, holiday-event, vegetarian, winter, dietary, christmas, seasonal, squash


## 3. Embed recipes

Using `all-MiniLM-L6-v2` — small, fast, 384-dim, good enough for V0.
This takes a few minutes on ~230K recipes; we cache the result.

In [5]:
from sentence_transformers import SentenceTransformer

EMBEDDING_CACHE = DATA_DIR / "recipe_embeddings_v0.npy"
MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

if EMBEDDING_CACHE.exists():
    print(f"Loading cached embeddings from {EMBEDDING_CACHE}")
    recipe_embeddings = np.load(EMBEDDING_CACHE)
else:
    print(f"Embedding {len(recipes):,} recipes...")
    recipe_embeddings = model.encode(
        recipes['text'].tolist(),
        show_progress_bar=True,
        batch_size=256,
        normalize_embeddings=True,
    )
    np.save(EMBEDDING_CACHE, recipe_embeddings)
    print(f"Saved to {EMBEDDING_CACHE}")

print(f"Shape: {recipe_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading cached embeddings from ../data/recipe_embeddings_v0.npy
Shape: (231637, 384)


## 4. Temporal train/test split

Split interactions by date — train on earlier, evaluate on later.
This avoids the leakage you'd get from random splitting.

In [6]:
interactions['date'] = pd.to_datetime(interactions['date'])
interactions = interactions.sort_values('date')

# Use an 80/20 temporal split
split_idx = int(len(interactions) * 0.8)
train_interactions = interactions.iloc[:split_idx]
test_interactions = interactions.iloc[split_idx:]

print(f"Train: {len(train_interactions):,} interactions")
print(f"  Date range: {train_interactions['date'].min()} → {train_interactions['date'].max()}")
print(f"Test: {len(test_interactions):,} interactions")
print(f"  Date range: {test_interactions['date'].min()} → {test_interactions['date'].max()}")

Train: 905,893 interactions
  Date range: 2000-01-25 00:00:00 → 2011-12-27 00:00:00
Test: 226,474 interactions
  Date range: 2011-12-27 00:00:00 → 2018-12-20 00:00:00


In [7]:
# Filter to positive interactions (rating >= 4) — these are "likes"
# Rating of 0 means no rating given (just a review), treat as neutral
train_positive = train_interactions[train_interactions['rating'] >= 4].copy()
test_positive = test_interactions[test_interactions['rating'] >= 4].copy()

print(f"Positive train interactions: {len(train_positive):,}")
print(f"Positive test interactions: {len(test_positive):,}")

# Only evaluate users who appear in both train and test
common_users = set(train_positive['user_id']) & set(test_positive['user_id'])
print(f"Users in both train and test: {len(common_users):,}")

Positive train interactions: 822,501
Positive test interactions: 181,223
Users in both train and test: 10,000


## 5. Build user embeddings

Each user's embedding = normalized mean of their positively-rated recipe embeddings.
Simple weighted mean for V0 — no recency weighting yet.

In [8]:
# Build recipe_id → index mapping for fast lookup
recipe_id_to_idx = dict(zip(recipes['id'], range(len(recipes))))

def build_user_embedding(user_recipe_ids):
    """Mean of recipe embeddings for recipes the user liked."""
    indices = [recipe_id_to_idx[rid] for rid in user_recipe_ids if rid in recipe_id_to_idx]
    if not indices:
        return None
    emb = recipe_embeddings[indices].mean(axis=0)
    # Normalize to unit vector for cosine similarity
    emb = emb / np.linalg.norm(emb)
    return emb

In [9]:
# Build user embeddings from train set only
user_liked_recipes = train_positive.groupby('user_id')['recipe_id'].apply(list)

user_embeddings = {}
for user_id in common_users:
    if user_id in user_liked_recipes.index:
        emb = build_user_embedding(user_liked_recipes[user_id])
        if emb is not None:
            user_embeddings[user_id] = emb

print(f"Built embeddings for {len(user_embeddings):,} users")

Built embeddings for 10,000 users


## 6. Retrieval via cosine similarity

For each user, find the top-k most similar recipes by dot product
(embeddings are already normalized, so dot product = cosine similarity).
231K vectors is small enough for numpy — no need for FAISS in V0.

In [10]:
K = 10

user_ids = list(user_embeddings.keys())
user_emb_matrix = np.array([user_embeddings[uid] for uid in user_ids], dtype=np.float32)
recipe_emb_matrix = recipe_embeddings.astype(np.float32)

# dot product of (10K users x 384) @ (384 x 231K) = (10K x 231K) similarity matrix
# Process in batches to keep memory reasonable
BATCH = 500
recipe_ids_arr = recipes['id'].values
recommendations = {}

for start in range(0, len(user_ids), BATCH):
    batch_embs = user_emb_matrix[start:start + BATCH]
    sims = batch_embs @ recipe_emb_matrix.T  # (batch x 231K)
    top_k_idx = np.argpartition(-sims, K, axis=1)[:, :K]
    # Sort within top-k by score
    for i in range(len(batch_embs)):
        idx = top_k_idx[i]
        sorted_idx = idx[np.argsort(-sims[i, idx])]
        uid = user_ids[start + i]
        recommendations[uid] = recipe_ids_arr[sorted_idx].tolist()

print(f"Generated top-{K} recommendations for {len(recommendations):,} users")

Generated top-10 recommendations for 10,000 users


## 7. Evaluation

Metrics on the temporal holdout:
- **Recall@K**: fraction of test-set liked recipes that appear in top-K
- **Hit Rate@K**: fraction of users who got at least one test-set hit in top-K
- **nDCG@K**: position-aware ranking quality

In [11]:
def recall_at_k(recommended, relevant):
    """What fraction of relevant items appear in the recommendation list?"""
    if not relevant:
        return 0.0
    return len(set(recommended) & set(relevant)) / len(relevant)

def hit_rate(recommended, relevant):
    """Did at least one relevant item appear?"""
    return 1.0 if set(recommended) & set(relevant) else 0.0

def ndcg_at_k(recommended, relevant):
    """Normalized discounted cumulative gain."""
    dcg = 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)  # i+2 because positions are 1-indexed
    # Ideal DCG: all relevant items at the top
    ideal_hits = min(len(relevant), len(recommended))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [12]:
# Build ground truth: for each user, what recipes did they like in the test set?
test_user_likes = test_positive.groupby('user_id')['recipe_id'].apply(set).to_dict()

recalls = []
hits = []
ndcgs = []

for uid in user_ids:
    relevant = test_user_likes.get(uid, set())
    if not relevant:
        continue
    recs = recommendations[uid]
    recalls.append(recall_at_k(recs, relevant))
    hits.append(hit_rate(recs, relevant))
    ndcgs.append(ndcg_at_k(recs, relevant))

print(f"Evaluated on {len(recalls):,} users")
print(f"")
print(f"Recall@{K}:   {np.mean(recalls):.4f}")
print(f"Hit Rate@{K}: {np.mean(hits):.4f}")
print(f"nDCG@{K}:     {np.mean(ndcgs):.4f}")

Evaluated on 10,000 users

Recall@10:   0.0009
Hit Rate@10: 0.0026
nDCG@10:     0.0006


## 8. Sanity checks

In [13]:
# Pick a random user and look at their recommendations vs history
sample_uid = user_ids[42]
print(f"User {sample_uid}")
print(f"\nLiked in train ({len(user_liked_recipes[sample_uid])} recipes):")
train_rids = user_liked_recipes[sample_uid][:5]
for rid in train_rids:
    if rid in recipe_id_to_idx:
        print(f"  - {recipes.iloc[recipe_id_to_idx[rid]]['name']}")

print(f"\nTop-{K} recommendations:")
for rid in recommendations[sample_uid]:
    idx = recipe_id_to_idx.get(rid)
    if idx is not None:
        print(f"  - {recipes.iloc[idx]['name']}")

test_likes = test_user_likes.get(sample_uid, set())
if test_likes:
    print(f"\nActually liked in test ({len(test_likes)} recipes):")
    for rid in list(test_likes)[:5]:
        if rid in recipe_id_to_idx:
            print(f"  - {recipes.iloc[recipe_id_to_idx[rid]]['name']}")

User 163986

Liked in train (64 recipes):
  - cabbage roll taste alike
  - honey mustard glaze  so silly i probably shouldn t post it
  - peanut butter and pickle sandwiches
  - rainy day yellow cake
  - carrot fruitcake

Top-10 recommendations:
  - cheese and squeeze   cheddar and beef   biscuit balls
  - caramelized onion and white bean flatbread
  - sausage and roasted peppers pasta bake
  - onion bread pudding
  - cheddar and veggie bread pudding
  - spinach beef biscuit bake
  - wonderful beef and noodle casserole
  - sweety and sour meatballs
  - beefy biscuit casserole
  - firecracker casserole

Actually liked in test (2 recipes):
  - kittencal s taco seasoning mix
  - sweet cheese ball


In [14]:
# Check: are we accidentally recommending recipes the user already saw?
leakage_counts = []
for uid in user_ids[:1000]:
    train_set = set(user_liked_recipes.get(uid, []))
    recs = set(recommendations[uid])
    leakage_counts.append(len(recs & train_set))

print(f"Mean train recipes in top-{K}: {np.mean(leakage_counts):.2f}")
print(f"This is expected — V0 doesn't filter seen items. Will fix if metrics look suspiciously high.")

Mean train recipes in top-10: 0.63
This is expected — V0 doesn't filter seen items. Will fix if metrics look suspiciously high.


## Notes

- If recall@10 is very low (<0.01), that's normal for this setup — the recipe space is huge (~230K) and users have few test interactions.
- If it's suspiciously high (>0.1), check for train/test leakage.
- Next steps: filter already-seen recipes from candidates, try recency weighting, experiment with different embedding models.